# LLM Guardrails for RAG Chatbots

## Introduction

Retrieval-Augmented Generation (RAG) systems are powerful tools for building knowledge-intensive LLM applications. However, integrating an LLM into a RAG pipeline introduces new challenges:

1.  **Hallucination**: The LLM might ignore the retrieved context or invent facts.
2.  **Output Formatting**: The LLM might not follow the expected output format (JSON, specific fields, etc.).
3.  **Safety & Security**: The system needs protection against adversarial inputs.

To address these, we use **Guardrails**. Guardrails are safety controls that validate LLM outputs against a defined schema.

In this notebook, we will use the `guardrails-ai` library with **Pydantic models** to enforce output structure validation for a RAG chatbot.

### Learning Objectives
1.  Understand the concept and necessity of Guardrails in RAG systems.
2.  Learn how to define a Guardrail using Pydantic models (modern approach).
3.  Implement a RAG guardrail that enforces output structure.
4.  Test the guardrail against valid and malformed inputs.
5.  Implement PII (Personally Identifiable Information) detection.

### How Guardrails Work: The Big Picture

Before we dive into code, let's understand where guardrails fit in a RAG system:

```
┌─────────────────────────────────────────────────────────────────────┐
│                        GUARDRAILS ARCHITECTURE                      │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   User Query    ──►  [INPUT GUARD]  ──►  LLM + RAG  ──►  [OUTPUT GUARD]  ──►  Safe Response
│                         │                                    │                     │
│                         │                                    │                     │
│                    Validates:                           Validates:            Returns:
│                    • Query length                       • JSON structure      • Validated data
│                    • No PII in input                    • Required fields     • Redacted PII
│                    • No prompt injection                • Type correctness    • Safe content
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
```

**Key Insight**: Guardrails act as "gatekeepers" on both sides of the LLM:
- **Input Guards**: Validate user queries before they reach the LLM
- **Output Guards**: Validate LLM responses before they reach the user

In this tutorial, we'll focus primarily on **output guardrails** (Steps 2-7), then briefly cover **input guardrails** (Step 9).

---

## Step 1: Setup and Installation

Before we can use guardrails, we need to install the required Python packages:

- **`guardrails-ai`**: The main library that provides validation functionality for LLM outputs.
- **`openai`**: The OpenAI Python client (we'll use this later for real LLM calls).
- **`pydantic`**: A data validation library that lets us define schemas as Python classes.

Run the cell below to install these packages.

In [1]:
!uv pip install guardrails-ai openai pydantic

Using Python 3.10.18 environment at: /Users/tarekatwan/Repos/MyWork/Teach/repos/advanced_machine_learning/.venv
Audited 3 packages in 110ms


---

## Step 2: Define the Output Schema with Pydantic

### What is a Schema?

A **schema** defines the expected structure of data. Think of it as a contract: "The LLM output MUST have these fields with these types."

### Why Pydantic?

**Pydantic** is a popular Python library for data validation. It lets us define schemas as regular Python classes, which is cleaner and more intuitive than XML or JSON schemas.

### Our RAG Response Schema

For our RAG chatbot, we want the LLM to return responses with:

| Field | Type | Description |
|-------|------|-------------|
| `answer` | string | The main response to the user's question |
| `sources` | list of strings | Document IDs that were used to generate the answer |
| `is_on_topic` | boolean | Was the question related to our allowed topic? |
| `confidence` | float (optional) | How confident is the model in its answer (0-1)? |

The code below creates this schema using Pydantic's `BaseModel` class.

In [3]:
from pydantic import BaseModel, Field
from typing import List, Optional

class RAGResponse(BaseModel):
    """Schema for RAG chatbot responses"""
    answer: str = Field(
        description="The final answer to the user's question, grounded in the context."
    )
    sources: List[str] = Field(
        default_factory=list,
        description="A list of source document IDs used to generate the answer."
    )
    is_on_topic: bool = Field(
        default=True,
        description="Whether the question was related to the allowed topic."
    )
    confidence: Optional[float] = Field(
        default=None,
        description="Confidence score between 0 and 1."
    )

print("RAGResponse schema defined successfully!")
print(f"Required fields: {list(RAGResponse.model_fields.keys())}")

RAGResponse schema defined successfully!
Required fields: ['answer', 'sources', 'is_on_topic', 'confidence']


### Understanding Validation Failure Strategies

When a guardrail detects invalid output, what should happen? The `guardrails-ai` library supports different **failure strategies**:

| Strategy | Behavior | When to Use |
|----------|----------|-------------|
| `"exception"` | Raises an error, stops execution | Critical violations (toxic content, security issues) |
| `"fix"` | Attempts to auto-correct the output | Minor formatting issues, PII redaction |
| `"filter"` | Removes violating items from lists | Content filtering in batch responses |
| `"refrain"` | Returns a generic safe response | When unsure, prefer safety |
| `"noop"` | Logs a warning but continues | Monitoring/debugging mode |

> **Note**: In this tutorial, we'll use basic Pydantic validation which raises exceptions on failure. The strategies above apply when using Guardrails Hub validators (covered in advanced tutorials).

For our Pydantic-based approach, we'll handle failures explicitly using try/except blocks (see Step 8).

---

## Step 3: Create the Guard

### What is a Guard?

A **Guard** is the core concept in `guardrails-ai`. It acts as a validator that:

1. **Parses** the LLM output (usually JSON text → Python dictionary)
2. **Validates** it against our schema (checks types, required fields, etc.)
3. **Returns** either:
   - ✅ A validated Python object if everything is correct
   - ❌ An error or `validation_passed=False` if something is wrong

### How to Create a Guard

We use `Guard.for_pydantic(YourModel)` to create a guard from our Pydantic schema.

```python
guard = Guard.for_pydantic(RAGResponse)
```

This tells the guard: "Any LLM output I receive should match the `RAGResponse` structure."

In [4]:
from guardrails import Guard

# Create a guard from the Pydantic model
guard = Guard.for_pydantic(RAGResponse)

print("Guard initialized successfully!")
print(f"Guard will validate outputs against: {RAGResponse.__name__}")

Guard initialized successfully!
Guard will validate outputs against: RAGResponse


---

## Step 4: Create a Mock LLM

### Why Mock?

Before testing with a real LLM (like GPT-4), we create a **mock function** that simulates LLM responses. This lets us:

- ✅ Test our guardrails without an API key
- ✅ Test without incurring API costs
- ✅ Simulate specific scenarios (valid, invalid, malformed outputs)

### What the Mock Does

Our `mock_llm_call()` function looks at the question and returns different responses:

| Question Contains | Response Type | Purpose |
|-------------------|---------------|----------|
| "metrics" | Valid JSON, on-topic | Test normal operation |
| "capital" | Valid JSON, off-topic | Test topic detection |
| "malformed" | Plain text (not JSON) | Test error handling |
| "missing" | JSON missing `answer` field | Test required field validation |

In [5]:
def mock_llm_call(question: str) -> str:
    """
    Simulates an LLM response based on the question content.
    In a real application, this would call OpenAI, Anthropic, etc.
    """
    if "metrics" in question.lower():
        # Simulate a good, on-topic response
        return '''{
            "answer": "The primary metrics for RAG evaluation are Faithfulness, Answer Relevancy, and Contextual Precision.",
            "sources": ["doc_1", "doc_2"],
            "is_on_topic": true,
            "confidence": 0.95
        }'''
    elif "capital" in question.lower():
        # Simulate an off-topic but correctly formatted response
        return '''{
            "answer": "I am a RAG chatbot for LLM topics. I cannot answer geography questions.",
            "sources": [],
            "is_on_topic": false,
            "confidence": 0.1
        }'''
    elif "malformed" in question.lower():
        # Simulate broken JSON (LLM failure)
        return "I am sorry, I cannot answer that in JSON format."
    elif "missing" in question.lower():
        # Simulate valid JSON but missing required 'answer' field
        return '''{
            "sources": ["doc_3"],
            "is_on_topic": true
        }'''
    else:
        # Generic valid response
        return '''{
            "answer": "I can answer questions about RAG and Guardrails.",
            "sources": ["intro_doc"],
            "is_on_topic": true
        }'''

print("Mock LLM function defined.")

Mock LLM function defined.


---

## Step 5: Testing the Guardrails

Now we'll test our guard against different scenarios to see how it handles:
1. Valid, on-topic responses
2. Off-topic but validly formatted responses
3. Malformed outputs (not valid JSON)
4. Outputs missing required fields

First, let's define our test questions:

In [6]:
# Define test questions - each will trigger a different mock response
questions = [
    "What are the primary metrics for RAG evaluation?",  # Triggers: Good response
    "What is the capital of France?",                   # Triggers: Off-topic
    "Generate a malformed response",                    # Triggers: Invalid JSON
    "Give me a response with missing fields",           # Triggers: Missing 'answer'
]

### Test Case 1: Valid On-Topic Response

**Scenario**: The user asks a question about RAG metrics. The LLM returns properly formatted JSON with all required fields.

**Expected Outcome**: ✅ Validation should PASS

In [7]:
print("=" * 60)
print("TEST CASE 1: Valid On-Topic Response")
print("=" * 60)

question = questions[0]
print(f"Question: {question}\n")

# Step 1: Get response from LLM (mock)
raw_output = mock_llm_call(question)
print(f"Raw LLM Output:\n{raw_output}\n")

# Step 2: Validate using the guard
result = guard.parse(raw_output)

# Step 3: Check result
print(f"✅ Validation Passed: {result.validation_passed}")
print(f"Validated Output (Python object):")
print(f"  - Answer: {result.validated_output['answer']}")
print(f"  - Sources: {result.validated_output['sources']}")
print(f"  - On Topic: {result.validated_output['is_on_topic']}")
print(f"  - Confidence: {result.validated_output['confidence']}")

TEST CASE 1: Valid On-Topic Response
Question: What are the primary metrics for RAG evaluation?

Raw LLM Output:
{
            "answer": "The primary metrics for RAG evaluation are Faithfulness, Answer Relevancy, and Contextual Precision.",
            "sources": ["doc_1", "doc_2"],
            "is_on_topic": true,
            "confidence": 0.95
        }

✅ Validation Passed: True
Validated Output (Python object):
  - Answer: The primary metrics for RAG evaluation are Faithfulness, Answer Relevancy, and Contextual Precision.
  - Sources: ['doc_1', 'doc_2']
  - On Topic: True
  - Confidence: 0.95


/Users/tarekatwan/Repos/MyWork/Teach/repos/advanced_machine_learning/.venv/lib/python3.10/site-packages/guardrails/validator_service/__init__.py:84: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


### Test Case 2: Off-Topic but Valid Format

**Scenario**: The user asks about geography (off-topic). The LLM correctly returns JSON but indicates `is_on_topic: false`.

**Expected Outcome**: ✅ Validation should PASS (format is correct)

**Key Insight**: Guardrails validate **structure**, not **semantics**. The `is_on_topic` field lets your application logic handle off-topic responses.

In [8]:
print("=" * 60)
print("TEST CASE 2: Off-Topic but Valid Format")
print("=" * 60)

question = questions[1]
print(f"Question: {question}\n")

raw_output = mock_llm_call(question)
print(f"Raw LLM Output:\n{raw_output}\n")

result = guard.parse(raw_output)

print(f"✅ Validation Passed: {result.validation_passed}")
print(f"Validated Output:")
print(f"  - Answer: {result.validated_output['answer']}")
print(f"  - On Topic: {result.validated_output['is_on_topic']}")
print(f"\n💡 Note: The guardrail passed because the FORMAT is valid.")
print(f"   We can use 'is_on_topic=False' to handle off-topic responses in our application.")

TEST CASE 2: Off-Topic but Valid Format
Question: What is the capital of France?

Raw LLM Output:
{
            "answer": "I am a RAG chatbot for LLM topics. I cannot answer geography questions.",
            "sources": [],
            "is_on_topic": false,
            "confidence": 0.1
        }

✅ Validation Passed: True
Validated Output:
  - Answer: I am a RAG chatbot for LLM topics. I cannot answer geography questions.
  - On Topic: False

💡 Note: The guardrail passed because the FORMAT is valid.
   We can use 'is_on_topic=False' to handle off-topic responses in our application.


### Handling Off-Topic Responses in Application Logic

The guardrail passed because the JSON format is valid, but the question was off-topic. In a real application, you would check `is_on_topic` and respond appropriately.

Here is a reusable function that demonstrates this pattern:

In [9]:
def process_rag_response(raw_llm_output: str) -> dict:
    """
    Process RAG response with topic filtering.
    
    This function demonstrates how to:
    1. Validate the JSON structure
    2. Check if the response is on-topic
    3. Return appropriate response or error message
    """
    # Step 1: Validate structure
    result = guard.parse(raw_llm_output)
    
    if not result.validation_passed:
        return {
            'status': 'error',
            'message': 'Invalid response format from LLM',
            'answer': None
        }
    
    # Step 2: Check if on-topic
    output = result.validated_output
    
    if not output['is_on_topic']:
        # Handle off-topic response
        return {
            'status': 'off_topic',
            'message': "I can only answer questions about LLM Guardrails and RAG. Please ask a relevant question.",
            'answer': None,
            'original_answer': output['answer']  # Keep original for logging
        }
    
    # Step 3: Return successful response
    return {
        'status': 'success',
        'message': None,
        'answer': output['answer'],
        'sources': output['sources'],
        'confidence': output['confidence']
    }

# Test with on-topic question
print("=" * 60)
print("Example 1: On-Topic Question")
print("=" * 60)
on_topic_response = mock_llm_call("What are the metrics for RAG?")
result1 = process_rag_response(on_topic_response)
print(f"Status: {result1['status']}")
print(f"Answer: {result1['answer']}")

print("\n" + "=" * 60)
print("Example 2: Off-Topic Question (Geography)")
print("=" * 60)
off_topic_response = mock_llm_call("What is the capital of France?")
result2 = process_rag_response(off_topic_response)
print(f"Status: {result2['status']}")
print(f"Message to User: {result2['message']}")
print(f"Original LLM Answer (for logging): {result2['original_answer']}")

print("\n" + "="*60)
print("Key Insight: The 'is_on_topic' field lets YOUR application")
print("decide how to handle off-topic queries!")
print("="*60)


Example 1: On-Topic Question
Status: success
Answer: The primary metrics for RAG evaluation are Faithfulness, Answer Relevancy, and Contextual Precision.

Example 2: Off-Topic Question (Geography)
Status: off_topic
Message to User: I can only answer questions about LLM Guardrails and RAG. Please ask a relevant question.
Original LLM Answer (for logging): I am a RAG chatbot for LLM topics. I cannot answer geography questions.

Key Insight: The 'is_on_topic' field lets YOUR application
decide how to handle off-topic queries!


### Test Case 3: Malformed Output (Invalid JSON)

**Scenario**: The LLM fails to follow instructions and returns plain text instead of JSON.

**Expected Outcome**: ❌ Validation should FAIL

**Why This Matters**: LLMs sometimes "break character" and don't follow the expected format. Guardrails catch these failures before they crash your application.

In [10]:
print("=" * 60)
print("TEST CASE 3: Malformed Output (Invalid JSON)")
print("=" * 60)

question = questions[2]
print(f"Question: {question}\n")

raw_output = mock_llm_call(question)
print(f"Raw LLM Output:\n{raw_output}\n")

try:
    result = guard.parse(raw_output)
    print(f"Validation Passed: {result.validation_passed}")
    if not result.validation_passed:
        print(f"❌ Validation Failed (as expected)")
        print(f"   Error: Output is not valid JSON")
except Exception as e:
    print(f"❌ Guardrail caught an error: {type(e).__name__}")
    print(f"   Message: {str(e)[:100]}...")

TEST CASE 3: Malformed Output (Invalid JSON)
Question: Generate a malformed response

Raw LLM Output:
I am sorry, I cannot answer that in JSON format.

Validation Passed: False
❌ Validation Failed (as expected)
   Error: Output is not valid JSON


### Test Case 4: Missing Required Field

**Scenario**: The LLM returns valid JSON, but forgets to include the `answer` field.

**Expected Outcome**: ❌ Validation should FAIL

**Why This Matters**: Required fields ensure your application always has the data it needs.

In [11]:
print("=" * 60)
print("TEST CASE 4: Missing Required Field")
print("=" * 60)

question = questions[3]
print(f"Question: {question}\n")

raw_output = mock_llm_call(question)
print(f"Raw LLM Output:\n{raw_output}\n")

try:
    result = guard.parse(raw_output)
    print(f"Validation Passed: {result.validation_passed}")
    if not result.validation_passed:
        print(f"❌ Validation Failed (as expected)")
        print(f"   Reason: Missing required 'answer' field")
except Exception as e:
    print(f"❌ Guardrail caught an error: {type(e).__name__}")
    print(f"   Message: {str(e)[:150]}...")

TEST CASE 4: Missing Required Field
Question: Give me a response with missing fields

Raw LLM Output:
{
            "sources": ["doc_3"],
            "is_on_topic": true
        }

Validation Passed: False
❌ Validation Failed (as expected)
   Reason: Missing required 'answer' field


---

## Step 6: Using with a Real LLM (Optional)

### What This Section Does

Up until now, we've been using a **mock function** that returns pre-defined responses. This section shows how to use guardrails with a **real LLM** (OpenAI's GPT-4).

### The Workflow

1. **Load API Key**: We load the OpenAI API key from a `.env` file.
2. **Create a Prompt**: We ask the LLM to respond in JSON format matching our schema.
3. **Call the LLM**: We send the prompt to GPT-4-mini.
4. **Validate the Response**: We use our guardrail to validate the LLM's output.

### Why This Matters

In production, you'll use real LLMs. This shows:
- How to instruct an LLM to output structured JSON
- How guardrails catch formatting issues even from real LLMs

### Prerequisites

To run this section, you need:
1. An OpenAI API key
2. A `.env` file with `OPENAI_API_KEY=sk-...`

If you don't have an API key, the cell will simply skip this test.

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

# Check if API key is available
if os.getenv("OPENAI_API_KEY"):
    from openai import OpenAI
    
    # Initialize the OpenAI client
    client = OpenAI()
    
    # Create a prompt that instructs the LLM to output JSON
    # Note: We explicitly tell the LLM the expected format
    prompt = f"""
You are a RAG chatbot about LLM Guardrails and Evaluation.
Answer the following question in JSON format with these fields:
- answer: Your response
- sources: List of source IDs (can be empty)
- is_on_topic: true if question is about LLM/RAG topics, false otherwise
- confidence: A number between 0 and 1

Question: What are guardrails in the context of LLMs?

Respond ONLY with valid JSON:
"""
    
    # Call OpenAI's GPT-4-mini model with JSON mode enabled
    # JSON mode ensures the LLM always returns valid JSON
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},  # 👈 NEW: Force JSON output
        temperature=0  # Low temperature = more deterministic output
    )
    
    # Get the raw text output from the LLM
    raw_output = response.choices[0].message.content
    print(f"Raw LLM Output:\n{raw_output}\n")
    
    # Validate the LLM output using our guardrail
    result = guard.parse(raw_output)
    print(f"Validation Passed: {result.validation_passed}")
    print(f"Validated Output: {result.validated_output}")
else:
    print("⚠️ OpenAI API Key not set. Skipping real LLM test.")
    print("To test with a real LLM:")
    print("  1. Create a file named '.env' in the same directory")
    print("  2. Add this line: OPENAI_API_KEY=sk-your-key-here")
    print("  3. Re-run this cell")

Raw LLM Output:
{
  "answer": "Guardrails in the context of LLMs (Large Language Models) refer to the safety measures and constraints implemented to ensure that the model's outputs are appropriate, reliable, and aligned with ethical standards. These can include content filters, guidelines for acceptable responses, and mechanisms to prevent harmful or biased outputs.",
  "sources": [],
  "is_on_topic": true,
  "confidence": 0.9
}

Validation Passed: True
Validated Output: {'answer': "Guardrails in the context of LLMs (Large Language Models) refer to the safety measures and constraints implemented to ensure that the model's outputs are appropriate, reliable, and aligned with ethical standards. These can include content filters, guidelines for acceptable responses, and mechanisms to prevent harmful or biased outputs.", 'sources': [], 'is_on_topic': True, 'confidence': 0.9}


/Users/tarekatwan/Repos/MyWork/Teach/repos/advanced_machine_learning/.venv/lib/python3.10/site-packages/guardrails/validator_service/__init__.py:84: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


### Why Use JSON Mode?

Notice the `response_format={"type": "json_object"}` parameter above. This is OpenAI's **JSON mode** which:

1. **Guarantees valid JSON** - The LLM will always return syntactically correct JSON
2. **Reduces parsing errors** - No more "LLM returned plain text instead of JSON" failures

> **Important**: JSON mode ensures *valid JSON*, but **not** that the JSON matches your schema. That's why we still need guardrails!

| Layer | What It Catches |
|-------|----------------|
| JSON Mode | Malformed JSON (syntax errors) |
| Guardrails | Missing fields, wrong types, invalid values |

---

## Step 7: PII Detection Guardrail

### What is PII?

**PII (Personally Identifiable Information)** is any data that could identify a specific individual. Examples include:

| PII Type | Examples |
|----------|----------|
| Email addresses | john.doe@example.com |
| Phone numbers | +1-555-123-4567 |
| Social Security Numbers | 123-45-6789 |
| Credit card numbers | 4111-1111-1111-1111 |
| Physical addresses | 123 Main St, City, State |

### Why Detect PII in LLM Outputs?

LLMs can accidentally **leak sensitive information** in their responses:

1. **Training data leakage**: The model might have memorized PII from its training data.
2. **Context window leakage**: In RAG systems, the retrieved documents might contain PII that gets included in the response.
3. **Compliance requirements**: Regulations like GDPR and HIPAA require protecting personal data.

### Learning Exercise: Manual PII Detection

Before using production tools, let's understand **what PII detection does under the hood**. Below is a simplified regex-based approach:

> **⚠️ Limitations of the Manual Approach:**
> - **Fragile regex patterns** - Won't catch international phone formats, unusual email domains
> - **No context awareness** - Emails in code examples get flagged as PII
> - **Maintenance burden** - You must update patterns for new PII types
> - **No obfuscation handling** - Can't detect "john [at] example [dot] com"
>
> **Production systems** use ML-based tools like [Microsoft Presidio](https://github.com/microsoft/presidio) or cloud services for higher accuracy.

In [13]:
import re

def detect_pii(text: str) -> dict:
    """
    Detect common PII patterns in text using regular expressions.
    
    Returns a dictionary with:
    - 'has_pii': True if any PII was detected
    - 'pii_types': List of PII types found
    - 'redacted_text': Text with PII replaced by [REDACTED]
    """
    pii_patterns = {
        'email': r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        'phone': r'(\+?1?[-.]?)?\(?\d{3}\)?[-.]?\d{3}[-.]?\d{4}',
        'ssn': r'\d{3}-\d{2}-\d{4}',
        'credit_card': r'\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}',
    }
    
    found_pii = []
    redacted_text = text
    
    for pii_type, pattern in pii_patterns.items():
        if re.search(pattern, text):
            found_pii.append(pii_type)
            redacted_text = re.sub(pattern, f'[REDACTED_{pii_type.upper()}]', redacted_text)
    
    return {
        'has_pii': len(found_pii) > 0,
        'pii_types': found_pii,
        'redacted_text': redacted_text
    }

print("PII detection function defined!")

PII detection function defined!


### Testing PII Detection

Let's test our PII detection on sample LLM outputs - one clean and one with PII.

In [14]:
# Sample LLM outputs to test
clean_output = "Guardrails are safety controls for LLMs that validate outputs against schemas."

pii_output = """According to our records, the customer John Smith can be reached at 
john.smith@company.com or by phone at 555-123-4567. His SSN is 123-45-6789."""

print("=" * 60)
print("TEST: Clean Output (No PII)")
print("=" * 60)
result1 = detect_pii(clean_output)
print(f"Original: {clean_output}")
print(f"Has PII: {result1['has_pii']}")
print(f"PII Types Found: {result1['pii_types']}")

print("\n" + "=" * 60)
print("TEST: Output with PII")
print("=" * 60)
result2 = detect_pii(pii_output)
print(f"Original:\n{pii_output}")
print(f"\nHas PII: {result2['has_pii']}")
print(f"PII Types Found: {result2['pii_types']}")
print(f"\nRedacted Text:\n{result2['redacted_text']}")

TEST: Clean Output (No PII)
Original: Guardrails are safety controls for LLMs that validate outputs against schemas.
Has PII: False
PII Types Found: []

TEST: Output with PII
Original:
According to our records, the customer John Smith can be reached at 
john.smith@company.com or by phone at 555-123-4567. His SSN is 123-45-6789.

Has PII: True
PII Types Found: ['email', 'phone', 'ssn']

Redacted Text:
According to our records, the customer John Smith can be reached at 
[REDACTED_EMAIL] or by phone at [REDACTED_PHONE]. His SSN is [REDACTED_SSN].


### Combining PII Detection with Schema Validation

Now let's create a complete guardrail pipeline that:
1. Validates the JSON structure (using Pydantic)
2. Checks for PII in the answer field
3. Redacts any PII found

This is how you would build a production-ready guardrail system.

In [15]:
def validate_and_redact(raw_llm_output: str) -> dict:
    """
    Complete validation pipeline:
    1. Parse and validate JSON structure
    2. Check for PII in the answer
    3. Redact PII if found
    """
    # Step 1: Validate structure
    result = guard.parse(raw_llm_output)
    
    if not result.validation_passed:
        return {
            'success': False,
            'error': 'Schema validation failed',
            'output': None
        }
    
    # Step 2: Check for PII in the answer
    answer = result.validated_output['answer']
    pii_check = detect_pii(answer)
    
    # Step 3: Create final output
    final_output = result.validated_output.copy()
    if pii_check['has_pii']:
        final_output['answer'] = pii_check['redacted_text']
        final_output['pii_detected'] = pii_check['pii_types']
        final_output['pii_warning'] = '⚠️ PII was detected and redacted'
    
    return {
        'success': True,
        'has_pii': pii_check['has_pii'],
        'output': final_output
    }

# Test with a mock output containing PII
mock_output_with_pii = '''{
    "answer": "The user John Smith (email: john@example.com) asked about RAG metrics.",
    "sources": ["user_query_log"],
    "is_on_topic": true,
    "confidence": 0.8
}'''

print("=" * 60)
print("COMBINED GUARDRAIL: Schema + PII Detection")
print("=" * 60)
print(f"Input:\n{mock_output_with_pii}\n")

result = validate_and_redact(mock_output_with_pii)
print(f"Validation Success: {result['success']}")
print(f"PII Detected: {result['has_pii']}")
print(f"\nFinal Output:")
for key, value in result['output'].items():
    print(f"  {key}: {value}")

COMBINED GUARDRAIL: Schema + PII Detection
Input:
{
    "answer": "The user John Smith (email: john@example.com) asked about RAG metrics.",
    "sources": ["user_query_log"],
    "is_on_topic": true,
    "confidence": 0.8
}

Validation Success: True
PII Detected: True

Final Output:
  answer: The user John Smith (email: [REDACTED_EMAIL]) asked about RAG metrics.
  sources: ['user_query_log']
  is_on_topic: True
  confidence: 0.8
  pii_detected: ['email']


### Key Takeaways: PII Guardrails

| Aspect | What We Learned |
|--------|----------------|
| **Why** | LLMs can leak sensitive data from training or context |
| **Detection** | Regex patterns for emails, phones, SSNs, credit cards |
| **Response** | Redact PII before returning to user |
| **Production** | Use specialized tools like `presidio` or cloud APIs |

---

## Step 8: Error Handling & Graceful Degradation

### Why Error Handling Matters

In production systems, **guardrail validation will sometimes fail**. When it does, your application shouldn't crash. Instead, it should:

1. **Log the failure** for debugging and monitoring
2. **Return a safe fallback response** to the user
3. **Optionally retry** with a modified prompt

### The Defensive Validation Pattern

Below is a robust validation function that handles failures gracefully:

In [16]:
def safe_validate(raw_output: str, fallback_message: str = "I encountered an issue processing this request.") -> dict:
    """
    Safely validate LLM output with graceful error handling.
    
    Returns:
        - Validated output if successful
        - Fallback response if validation fails
    """
    try:
        result = guard.parse(raw_output)
        
        if result.validation_passed:
            return {
                'status': 'success',
                'output': result.validated_output
            }
        else:
            # Validation failed but didn't raise an exception
            print(f"⚠️ Validation failed (no exception)")
            return {
                'status': 'fallback',
                'output': {
                    'answer': fallback_message,
                    'sources': [],
                    'is_on_topic': True,
                    'confidence': None
                },
                'reason': 'Validation failed'
            }
            
    except Exception as e:
        # Log the error for debugging
        print(f"❌ Validation error: {type(e).__name__}: {str(e)[:100]}")
        return {
            'status': 'error',
            'output': {
                'answer': fallback_message,
                'sources': [],
                'is_on_topic': True,
                'confidence': None
            },
            'reason': str(e)
        }

print("safe_validate function defined!")

safe_validate function defined!


In [17]:
# Test the defensive validation with different scenarios
print("=" * 60)
print("TEST: Defensive Validation")
print("=" * 60)

# Test 1: Valid output
valid_output = '{"answer": "Guardrails validate LLM outputs.", "sources": ["doc1"], "is_on_topic": true}'
result1 = safe_validate(valid_output)
print(f"\n✅ Valid Output Test:")
print(f"   Status: {result1['status']}")
print(f"   Answer: {result1['output']['answer'][:50]}...")

# Test 2: Invalid JSON (would crash without try/except)
invalid_output = "This is not JSON at all"
result2 = safe_validate(invalid_output)
print(f"\n❌ Invalid JSON Test:")
print(f"   Status: {result2['status']}")
print(f"   Fallback Answer: {result2['output']['answer']}")

# Test 3: Missing required field
missing_field = '{"sources": ["doc1"], "is_on_topic": true}'  # Missing 'answer'
result3 = safe_validate(missing_field)
print(f"\n❌ Missing Field Test:")
print(f"   Status: {result3['status']}")
print(f"   Fallback Answer: {result3['output']['answer']}")

print("\n" + "=" * 60)
print("Key Insight: The application never crashes, even with bad input!")
print("=" * 60)

TEST: Defensive Validation

✅ Valid Output Test:
   Status: success
   Answer: Guardrails validate LLM outputs....
⚠️ Validation failed (no exception)

❌ Invalid JSON Test:
   Status: fallback
   Fallback Answer: I encountered an issue processing this request.
⚠️ Validation failed (no exception)

❌ Missing Field Test:
   Status: fallback
   Fallback Answer: I encountered an issue processing this request.

Key Insight: The application never crashes, even with bad input!


### Key Takeaways: Error Handling

| Scenario | Without Error Handling | With Error Handling |
|----------|----------------------|--------------------|
| Invalid JSON | ❌ Application crashes | ✅ Returns fallback response |
| Missing field | ❌ Application crashes | ✅ Returns fallback response |
| Valid output | ✅ Works normally | ✅ Works normally |

> **Production Tip**: In real systems, you would also:
> - Send failed validations to a logging service (e.g., DataDog, CloudWatch)
> - Track failure rates to detect model degradation
> - Implement retry logic with exponential backoff

---

## Step 9: Input Guardrails (Validating User Queries)

### The Other Side of the Coin

So far, we've focused on validating **LLM outputs**. But what about **user inputs**?

Input guardrails protect your system from:

| Threat | Description | Example |
|--------|-------------|----------|
| **Empty queries** | Waste of API calls | "" or "   " |
| **Excessively long queries** | Token limit issues, higher costs | 10,000 character query |
| **PII in queries** | Users accidentally sharing sensitive data | "My SSN is 123-45-6789" |
| **Prompt injection** | Malicious attempts to manipulate the LLM | "Ignore previous instructions..." |

### Basic Input Validation Function

In [18]:
def validate_user_input(query: str, max_length: int = 1000) -> dict:
    """
    Validate user input before sending to the LLM.
    
    Checks for:
    1. Empty or too short queries
    2. Excessively long queries  
    3. PII in user input (users shouldn't send sensitive data to LLMs)
    
    Returns:
        dict with 'valid' (bool) and either 'sanitized_query' or 'reason'
    """
    # Check 1: Empty or too short
    cleaned = query.strip()
    if len(cleaned) < 3:
        return {
            'valid': False,
            'reason': 'Query too short. Please provide more detail.'
        }
    
    # Check 2: Too long
    if len(cleaned) > max_length:
        return {
            'valid': False,
            'reason': f'Query too long ({len(cleaned)} chars). Maximum is {max_length}.'
        }
    
    # Check 3: PII in input (reuse our detect_pii function)
    pii_check = detect_pii(cleaned)
    if pii_check['has_pii']:
        return {
            'valid': False,
            'reason': f"Please remove {', '.join(pii_check['pii_types'])} from your query."
        }
    
    # All checks passed
    return {
        'valid': True,
        'sanitized_query': cleaned
    }

print("validate_user_input function defined!")

validate_user_input function defined!


In [19]:
# Test input validation with various scenarios
print("=" * 60)
print("TEST: Input Guardrails")
print("=" * 60)

test_inputs = [
    ("What are RAG evaluation metrics?", "Normal query"),
    ("", "Empty query"),
    ("Hi", "Too short"),
    ("My email is john@example.com and I want to know about RAG", "Contains PII"),
    ("A" * 1500, "Too long"),
]

for query, description in test_inputs:
    result = validate_user_input(query)
    status = "✅ Valid" if result['valid'] else f"❌ Invalid: {result['reason']}"
    print(f"\n{description}:")
    print(f"  Query: '{query[:40]}{'...' if len(query) > 40 else ''}'")
    print(f"  Result: {status}")

print("\n" + "=" * 60)
print("Key Insight: Validate inputs BEFORE calling the LLM to save costs!")
print("=" * 60)

TEST: Input Guardrails

Normal query:
  Query: 'What are RAG evaluation metrics?'
  Result: ✅ Valid

Empty query:
  Query: ''
  Result: ❌ Invalid: Query too short. Please provide more detail.

Too short:
  Query: 'Hi'
  Result: ❌ Invalid: Query too short. Please provide more detail.

Contains PII:
  Query: 'My email is john@example.com and I want ...'
  Result: ❌ Invalid: Please remove email from your query.

Too long:
  Query: 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...'
  Result: ❌ Invalid: Query too long (1500 chars). Maximum is 1000.

Key Insight: Validate inputs BEFORE calling the LLM to save costs!


### Complete Guardrail Pipeline

Now let's put it all together - a complete function that validates **both** input and output:

In [20]:
def complete_rag_pipeline(user_query: str) -> dict:
    """
    Complete RAG pipeline with input and output guardrails.
    
    Flow:
    1. Validate user input
    2. Call LLM (mock in this example)
    3. Validate LLM output
    4. Check for PII and redact
    5. Return safe response
    """
    print(f"📥 Received query: '{user_query[:50]}...'" if len(user_query) > 50 else f"📥 Received query: '{user_query}'")
    
    # Step 1: Input validation
    input_check = validate_user_input(user_query)
    if not input_check['valid']:
        print(f"🚫 Input rejected: {input_check['reason']}")
        return {'status': 'input_rejected', 'message': input_check['reason']}
    print("✅ Input validated")
    
    # Step 2: Call LLM (using mock)
    raw_output = mock_llm_call(input_check['sanitized_query'])
    print("🤖 LLM responded")
    
    # Step 3: Output validation with error handling
    validation_result = safe_validate(raw_output)
    if validation_result['status'] != 'success':
        print(f"⚠️ Output validation failed, using fallback")
        return {'status': 'fallback', 'output': validation_result['output']}
    print("✅ Output validated")
    
    # Step 4: PII check and redaction
    answer = validation_result['output']['answer']
    pii_check = detect_pii(answer)
    if pii_check['has_pii']:
        print(f"🔒 PII detected and redacted: {pii_check['pii_types']}")
        validation_result['output']['answer'] = pii_check['redacted_text']
    
    # Step 5: Return safe response
    print("📤 Returning safe response")
    return {'status': 'success', 'output': validation_result['output']}

# Test the complete pipeline
print("=" * 60)
print("COMPLETE PIPELINE TEST")
print("=" * 60)

result = complete_rag_pipeline("What are the primary metrics for RAG evaluation?")
print(f"\nFinal Result:")
print(f"  Status: {result['status']}")
print(f"  Answer: {result['output']['answer']}")

COMPLETE PIPELINE TEST
📥 Received query: 'What are the primary metrics for RAG evaluation?'
✅ Input validated
🤖 LLM responded
✅ Output validated
📤 Returning safe response

Final Result:
  Status: success
  Answer: The primary metrics for RAG evaluation are Faithfulness, Answer Relevancy, and Contextual Precision.


---

## 🎯 Student Challenge: Build a Complete Validation Pipeline

Now it's your turn! Using what you've learned, create a more sophisticated validation system.

### Task 1: Enhanced Input Validator

Modify `validate_user_input()` to also check for:
- **Minimum word count** (at least 2 words)
- **Suspicious patterns** like "ignore previous instructions" (basic prompt injection detection)

### Task 2: Enhanced Output Schema

Extend the `RAGResponse` Pydantic model to include:
```python
# Add these fields to RAGResponse:
reasoning: Optional[str] = Field(default=None, description="How the answer was derived")
word_count: int = Field(ge=10, le=500, description="Answer must be 10-500 words")
```

### Task 3: Test Edge Cases

Create test cases for:
1. A query that's exactly at the character limit
2. A query with an obfuscated email: "Contact me at john [at] example [dot] com"
3. An LLM response where `word_count` is outside the valid range

### Bonus Challenge

Research and implement a basic **prompt injection detector** using simple keyword matching. Consider patterns like:
- "Ignore previous instructions"
- "Disregard your guidelines"
- "Pretend you are"

> **Hint**: Look at how we implemented `detect_pii()` with regex patterns for inspiration!